[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/limorigu/uchicago-python-ai-workshop/blob/main/notebooks/03_news_dashboard_STARTER.ipynb)

# 03 · Case study 2 — A news screen for an investment target (Day 2)

**Situation.** Your team is evaluating **Marriott International** (NASDAQ: MAR) as a potential investment. Before the first call with management you want a quick, repeatable read of what the news says about the company — and, crucially, you want to know *when to trust that read.*

*(Why Marriott and not, say, Hilton? We tried Hilton first — and the feed drowned in headlines about Steve Hilton, who is running for governor of California next month. "Hilton" is also a person, an island, and a celebrity. That query ambiguity is a real lesson — you will meet it again below — but it makes a bad default target three weeks before an election.)*

**What you will build:** `fetch → clean → baseline sentiment → LLM classification → evaluate → iterate the prompt`.

| Part | When | Output |
|---|---|---|
| A. Fetch, clean, baseline | 10:30–11:10 | a DataFrame of headlines with a naive sentiment score, and a first look at where it is wrong |
| B. Gold labels + LLM run | 11:10–12:00 | your team's 20 hand labels, inter-annotator agreement, a first LLM classification |
| B (cont.) Evaluate + iterate | 1:10–1:40 | the accuracy table, the disagreement list, and a **required** `PROMPT_V3` with a logged score |

**Teamwork rule (teams of 5, finance + tech).** Tech students drive the code; finance students drive every *judgement* — what counts as relevant, what an investor calls negative, which disagreements matter. Rotate the keyboard at Part B. The sprint rubric this afternoon scores exactly this split.

**About `newslib`.** The pipeline's plumbing lives in `app/newslib.py` — roughly 300 lines of Python **written for this workshop**. It is *not* an installed package like `pandas` or `numpy`: it is a plain `.py` file sitting in this repository, which is why the setup cell adds the `app/` folder to Python's path before `import newslib` works. You are encouraged to open it and read it. It follows exactly the library-plus-tests pattern the finance cohort pushed through GitHub Actions yesterday with `case1/whitespace.py` — same idea, different case. Today nobody builds plumbing; you build judgement on top of it.

**No API key?** Every cell runs in *mock mode* (a rule-based stand-in). Your team can also run the prompt in a chat UI on 20 headlines by hand — see Part B.

**If the live feed is blocked** on the classroom network, the notebook automatically switches to a *synthetic* feed about a *fictional* hotel company, "Harborview Hotels", so the pipeline can still be exercised. A banner will tell you.

In [21]:
# --- Setup: get the workshop files into this Colab session ------------------------------
# Downloads the workshop repository (data + helper code). Takes ~10 seconds the first time.
import os, sys
REPO_URL = "https://github.com/limorigu/uchicago-python-ai-workshop"
if not any(os.path.exists(d) for d in ["workshop/data", "data", "../data"]):
    !GIT_TERMINAL_PROMPT=0 git clone -q --depth 1 $REPO_URL workshop
DATA = next((d for d in ["workshop/data", "data", "../data"] if os.path.exists(d)), None)
if DATA is None:
    raise RuntimeError("Could not download the workshop files. Check your internet connection and "
                       "that this link opens in your browser: " + REPO_URL + " . Still stuck? Email the instructor.")
sys.path.insert(0, "workshop/app" if os.path.exists("workshop/app") else "../app")
print("Data folder:", DATA, "| files:", sorted(os.listdir(DATA)))

Data folder: workshop/data | files: ['case1', 'case2']


## Part A · Step 1 — Fetch
Google News publishes an **RSS feed** for any search — a machine-readable version of the search results that anyone can download, no API key needed. RSS is written in **XML**, a text format (like HTML's stricter cousin) where each headline sits inside tags: `<item><title>…</title><source>…</source><pubDate>…</pubDate></item>`.

`newslib.fetch_google_news` does two things: builds the search URL, then walks the XML and pulls out one row per `<item>`. Remember: `newslib` is our workshop file, not a package — if you want to see the ~25 lines that do this, open `app/newslib.py` in the repo (or run `nl.fetch_google_news??` in a cell to print the source right here).

In [35]:
import pandas as pd
import newslib as nl

COMPANY = "Marriott"
DESCRIPTION = "global hotel franchisor, NASDAQ: MAR"
EXTRA = 'when:30d -library'      # Google search syntax: time window + exclusions (the Marriott Library is a thing)

df = nl.fetch_with_fallback(COMPANY, f"{DATA}/case2/news_sample_cache.csv", extra_terms=EXTRA)
if df.get("is_synthetic", pd.Series([False])).any():
    COMPANY, DESCRIPTION = "Harborview Hotels", "fictional global hotel franchisor (synthetic classroom feed)"
print(len(df), "headlines about", COMPANY)
df.head()

100 headlines about Marriott


,title,source,published,link,query,is_synthetic
0,"Marriott Bonvoy Guide: How to Earn, Redeem & Maximize Points",NerdWallet,2026-10-13 16:29:23+00:00,https://news.google.com/rss/articles/CBMiigFBVV95cUxQWVFNWWVqQmppWC1td3A1NEt5cDVMNFVIV0ExWFdBOERVTUdvMldxa0V5R1g1b2t...,"""Marriott"" when:30d -library",False
1,"Hilton, Marriott franchisee files for Chapter 11 bankruptcy",Yahoo Finance,2026-10-06 07:00:00+00:00,https://news.google.com/rss/articles/CBMipAFBVV95cUxNRllXZlhRSDZfTXBhaUZUUHByM1JLS29WMG5jMFBibDRMdVFhRXJHUkJLTHhhRGF...,"""Marriott"" when:30d -library",False
2,Marriott debuts Complimentary Upgrade List app feature and a new lounge experience,The Points Guy,2026-10-13 05:44:08+00:00,https://news.google.com/rss/articles/CBMinAFBVV95cUxPLU9lOFlldUJfdDdFcjlJcjhTdkJxdzByaUIwWTE5OG1fb2ZZNjhOanRhUjIxSXB...,"""Marriott"" when:30d -library",False
3,This New All-Inclusive Resort in Costa Rica Has 17 Pools and Lets Guests Roast Their Own Coffee,Travel + Leisure,2026-10-07 07:00:00+00:00,https://news.google.com/rss/articles/CBMixgFBVV95cUxPODhDRXh2aVA3UEdfMWVjanVtMDJLVTV1R1pCVHFRNGszWG9mU20wdTM2ZFNuYXJ...,"""Marriott"" when:30d -library",False
4,BYU opens basketball season with Marriott Madness,Deseret News,2026-10-07 07:00:00+00:00,https://news.google.com/rss/articles/CBMiuAFBVV95cUxNUWgxbUFYUUR4NnV2N3J5eDFPNURhdzB1Ny1jTjJYVkFRM00wcVVaVUwwcm9oaWV...,"""Marriott"" when:30d -library",False


**Look at the raw titles before doing anything else.** What is *not* about the company? What is duplicated? What would fool a keyword approach?

In [34]:
pd.set_option("display.max_colwidth", 120)
df[["published", "source", "title"]].head(25)

,published,source,title
0,2026-10-13 16:29:23+00:00,NerdWallet,"Marriott Bonvoy Guide: How to Earn, Redeem & Maximize Points"
1,2026-10-06 07:00:00+00:00,Yahoo Finance,"Hilton, Marriott franchisee files for Chapter 11 bankruptcy"
2,2026-10-13 05:44:08+00:00,The Points Guy,Marriott debuts Complimentary Upgrade List app feature and a new lounge experience
3,2026-10-07 07:00:00+00:00,Travel + Leisure,This New All-Inclusive Resort in Costa Rica Has 17 Pools and Lets Guests Roast Their Own Coffee
4,2026-10-07 07:00:00+00:00,Deseret News,BYU opens basketball season with Marriott Madness
5,2026-09-30 07:00:00+00:00,The Washington Post,Opinion | A grim prognosis for the post-literate future
6,2026-09-19 07:00:00+00:00,Legacy | Obituary,"Stephen Alexander Marriott Obituary (2026) - Salt Lake City, UT - Utah Simple Cremations"
7,2026-10-14 13:20:02+00:00,Life at Marriott Blog,Where Can a Food & Beverage Career Take You?
8,2026-10-06 07:00:00+00:00,PR Newswire,JW Marriott Debuts Its First All-Inclusive Resort with the Opening of JW Marriott Costa Elena Resort & Spa
9,2026-10-06 07:00:00+00:00,Axios,Detroit's new JW Marriott gears up for February debut


In [6]:
# OPTIONAL (live feed only): see why we abandoned Hilton as the target.
# df_hilton = nl.fetch_google_news("Hilton", extra_terms="when:7d")
# df_hilton.title.head(20)   # how many are about the hotel company vs the candidate for governor?

## Step 2 — Clean

Your TODO below calls `nl.clean_headlines(df)`. Before (or after) you run it, **figure out what the function actually does** — pick whichever of these you'd reach for on the job:

- **Ask the built-in AI.** Colab has Gemini built in (the ✨ *Ask Gemini* button, or the chat icon in the left sidebar): paste the function name and ask "what does `clean_headlines` in the open `newslib.py` do, step by step?" Or paste the function into whatever assistant you have open in another window (Claude, ChatGPT).
- **Read the source yourself.** Run `nl.clean_headlines??` in a cell (two question marks prints the full source), or open `app/newslib.py` via Colab's file browser (📁 icon in the left sidebar → `workshop/app/newslib.py`, double-click to view).

Then verify your understanding against the data: the row counts before/after *are* the answer key.

In [7]:
# TODO: call nl.clean_headlines(df) and compare row counts before/after. What did it remove?

In [36]:
original_df = df.copy()

cleaned_df = nl.clean_headlines(df)

print("Original:", len(original_df))
print("Cleaned:", len(cleaned_df))

print("Duplicate titles:",
      original_df["title"].duplicated().sum())

duplicates = original_df[
    original_df["title"].duplicated(keep=False)
]

print(duplicates[["title", "source"]])

Original: 100
Cleaned: 99
Duplicate titles: 1
                                                                                                         title  \
8   JW Marriott Debuts Its First All-Inclusive Resort with the Opening of JW Marriott Costa Elena Resort & Spa   
65  JW Marriott Debuts Its First All-Inclusive Resort with the Opening of JW Marriott Costa Elena Resort & Spa   

           source  
8     PR Newswire  
65  Yahoo Finance  


## Step 3 — Baseline: a naive lexicon
`newslib.lexicon_sentiment` counts positive and negative words from a small finance word list (real work would use the Loughran–McDonald dictionary). It is deliberately simple. **Your job is to find where it breaks.**

Expect a lot of neutrals on the live feed: real headlines hedge ("weighs", "expected to", "in talks") and the word list is small. That is itself a finding — a baseline that calls almost everything neutral can still score *high accuracy* if your gold labels are mostly neutral too, which is exactly why you will look at per-class precision/recall this afternoon, not just accuracy.

In [37]:
df = nl.add_lexicon_sentiment(df)
df["lex_label"].value_counts()

,count
lex_label,
neutral,80
positive,17
negative,3


In [42]:
# TODO: show the 10 most positive and 10 most negative headlines by lex_score. Do you agree with them?
# TODO: find at least 3 headlines where the lexicon is clearly wrong and write down WHY (negation? sarcasm? finance jargon?
#       a word that is positive in general English but negative for an investor?)
print("10 Most Positive Headlines")

df.nlargest(10, "lex_score")[["title", "lex_score"]]



10 Most Positive Headlines


,title,lex_score
4,BYU opens basketball season with Marriott Madness,0.143
23,Marriott opens debut all-inclusive JW Marriott resort,0.125
34,Spire Hospitality opens Alabama hotel under Series by Marriott,0.111
57,Marriott extended-stay hotel opens first Memphis-area location,0.111
89,JW Marriott Opens All-Inclusive Resort in Costa Rica,0.111
11,World’s First All-Inclusive JW Marriott Opens in Guanacaste,0.100
30,Marriott (MAR) Stock May Be Fully Priced Following Thailand Hotel Expansion,0.091
50,"Lodging Lowdown: IHG names global sales leader, Series by Marriott expands",0.091
63,City Express by Marriott opens first South Asia hotel in Rajasthan,0.091
97,Marriott vs. Hilton: MAR offers lower valuation and stronger revenue growth,0.091


In [43]:
print("10 Most Negative Headlines")

df.nsmallest(10, "lex_score")[["title", "lex_score"]]

10 Most Negative Headlines


,title,lex_score
70,Get 10K points for following the fairway to Williamsburg this fall.,-0.091
21,How Marriott’s leadership pipeline is helping it navigate Europe’s hospitality talent shortfall,-0.071
64,UCF Q&A: What’s the biggest reason for offensive struggles? What is QB Rocco Marriott’s future?,-0.056
0,"Marriott Bonvoy Guide: How to Earn, Redeem & Maximize Points",0.000
1,"Hilton, Marriott franchisee files for Chapter 11 bankruptcy",0.000
3,This New All-Inclusive Resort in Costa Rica Has 17 Pools and Lets Guests Roast Their Own Coffee,0.000
5,Opinion | A grim prognosis for the post-literate future,0.000
6,"Stephen Alexander Marriott Obituary (2026) - Salt Lake City, UT - Utah Simple Cremations",0.000
7,Where Can a Food & Beverage Career Take You?,0.000
8,JW Marriott Debuts Its First All-Inclusive Resort with the Opening of JW Marriott Costa Elena Resort & Spa,0.000


In [48]:
display(df[["title", "lex_score", "lex_label"]])

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(
    df[["title", "lex_score", "lex_label"]]
    .sort_values("lex_score", ascending=False)
)

,title,lex_score,lex_label
0,"Marriott Bonvoy Guide: How to Earn, Redeem & Maximize Points",0.000,neutral
1,"Hilton, Marriott franchisee files for Chapter 11 bankruptcy",0.000,neutral
2,Marriott debuts Complimentary Upgrade List app feature and a new lounge experience,0.083,positive
3,This New All-Inclusive Resort in Costa Rica Has 17 Pools and Lets Guests Roast Their Own Coffee,0.000,neutral
4,BYU opens basketball season with Marriott Madness,0.143,positive
5,Opinion | A grim prognosis for the post-literate future,0.000,neutral
6,"Stephen Alexander Marriott Obituary (2026) - Salt Lake City, UT - Utah Simple Cremations",0.000,neutral
7,Where Can a Food & Beverage Career Take You?,0.000,neutral
8,JW Marriott Debuts Its First All-Inclusive Resort with the Opening of JW Marriott Costa Elena Resort & Spa,0.000,neutral
9,Detroit's new JW Marriott gears up for February debut,0.000,neutral


,title,lex_score,lex_label
4,BYU opens basketball season with Marriott Madness,0.143,positive
23,Marriott opens debut all-inclusive JW Marriott resort,0.125,positive
34,Spire Hospitality opens Alabama hotel under Series by Marriott,0.111,positive
57,Marriott extended-stay hotel opens first Memphis-area location,0.111,positive
89,JW Marriott Opens All-Inclusive Resort in Costa Rica,0.111,positive
11,World’s First All-Inclusive JW Marriott Opens in Guanacaste,0.100,positive
30,Marriott (MAR) Stock May Be Fully Priced Following Thailand Hotel Expansion,0.091,positive
63,City Express by Marriott opens first South Asia hotel in Rajasthan,0.091,positive
50,"Lodging Lowdown: IHG names global sales leader, Series by Marriott expands",0.091,positive
97,Marriott vs. Hilton: MAR offers lower valuation and stronger revenue growth,0.091,positive


For example BYU basketball season with Marriot Madness has the highest lex score, but its completely irrelvant (but it has the positive word open) so the score is high.

Marriot Stock May be fully priced following Thailand Hotel expansion(already at full price, but reacted to the word expansion)
should not have a positive lex score

How Marriot Leadership pipeline is helping it navigate Europe hospitality talent shortfall("shortfall" is negative, but headline is emphasizing how its responding to the talent shortage.

## Part B · Step 4 — Hand labels (the gold set) — 15 min, team
Your team labels **15 headlines**: ten that *every* team labels (the common block) plus five that only your team covers. The cell below slices them out of the cleaned frame by your team number.

Each of the five of you labels all 15 **independently, without talking** (relevant / sentiment / category). Then compare: the share where you agree is your **inter-annotator agreement** — no model can be measured more precisely than that. Argue out a final team label for each headline.

**Class pool:** one person per team enters your 15 agreed rows into the shared class sheet — **the link is on the Case 2 slide being projected right now.** The design: 10 teams × the same common block = ten independent labels for each of those ten headlines (that's the class-wide agreement number for the debrief), and 10 teams × 5 unique = 50 more headlines of coverage. By lunch the room owns a ~60-headline gold set with a measured ceiling.

Categories: `earnings, m&a_or_capital, management, demand_or_macro, legal_or_regulatory, product_or_brand, esg_or_labor, other`.
Sentiment is **from an investor's perspective**: a hotel opening is routine → neutral; a CEO departure → usually negative.

In [49]:
TEAM = 5   # <-- your team number (1-10, from the teams slide)

# Deterministic assignment: sort alphabetically so every team slices the same deck.
pool = df.sort_values("title").reset_index(drop=True)
common = pool.iloc[:10]                                  # every team labels these ten
mine = pool.iloc[10 + (TEAM - 1) * 5 : 10 + TEAM * 5]    # five more, unique to your team
to_label = pd.concat([common, mine])
to_label_out = to_label.reindex(columns=["id", "title"]).assign(relevant="", sentiment="", category="")
to_label_out.to_csv("team_labels_blank.csv", index=False)
print("Wrote team_labels_blank.csv —", len(to_label), "headlines. Each of you labels all of them, separately.")
to_label[["title"]]

Wrote team_labels_blank.csv — 15 headlines. Each of you labels all of them, separately.


,title
0,95% of Marriott hotels surveyed call local dishes the top dining trend.
1,A TakeCare Champion in Panama on Prioritizing People
2,"Aimbridge Hospitality Assumes Management of The Westin Cincinnati, Named Future Operator of Portman’s Cincinnati Downtown Marriott"
3,"As Da Nang's MICE Ambitions Rise, Courtyard by Marriott Danang Han River Puts a New Spin on Business Events"
4,"BYU men’s, women’s basketball 'play for each other'"
5,BYU opens basketball season with Marriott Madness
6,Beijing Marriott Marquis Hotel Hosts China Open Delegation During Its Opening Weeks
7,Bourbon on 33 brings a weekly chocolate pop-up to JW Marriott downtown
8,City Express by Marriott Debuts in India with the Opening of City Express by Marriott Udaipur Nathdwara Highway
9,City Express by Marriott opens first South Asia hotel in Rajasthan


In [11]:
# After the team has argued out one agreed label per headline, save the result as our_labels.csv
# and upload it to this Colab session (folder icon on the left -> upload). Then re-run this cell.
import os
gold_path = "our_labels.csv"
if os.path.exists(gold_path):
    gold = pd.read_csv(gold_path)
    key = "id" if "id" in df.columns and "id" in gold.columns else "title"
    m = df.merge(gold[[key, "relevant", "sentiment", "category"]], on=key, how="inner", suffixes=("", "_gold"))
    print(len(m), "labelled headlines")
else:
    print("No labels yet — label your 15 headlines, save as our_labels.csv, upload, re-run.")

No labels yet — label your 15 headlines, save as our_labels.csv, upload, re-run.


In [12]:
# If two group members labelled separately, measure agreement:
# a = pd.read_csv("labels_alice.csv"); b = pd.read_csv("labels_bob.csv")
# print("human-human agreement on sentiment:", nl.agreement_rate(a.sentiment, b.sentiment))

## Step 5 — LLM classification
`newslib.PROMPT_V1` is what most people write first. `PROMPT_V2` applies this morning's **context-engineering** lesson — every line in it earned its place in the window: a role and an investor's perspective (so the model knows *whose* judgement to apply), an explicit relevance check (the exit ramp for headlines about the wrong Marriott), a closed category list (no invented labels), a JSON schema (so code can read the answer), and a rationale field (so *you* can read the model's reasoning when it disagrees with you).

**Track A (you brought your own API key — via your university's AI access):** set `ANTHROPIC_API_KEY` in Colab's *Secrets* panel (🔑 icon) — never paste it in a cell — and create the client below. It's your key and your spend — worth batching calls and using a cheaper model while you iterate.
**Track B (no key):** leave `client = None` (mock mode), *and* have one group member paste `PROMPT_V2` + 20 headlines into a chat UI, record the JSON answers into your labels file as `llm_sentiment`, and compare by hand.

In [13]:
client = None
try:
    from google.colab import userdata          # Colab secrets
    # .strip(): a stray newline from the paste would make every API call fail
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
except Exception:
    pass
if os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = os.environ["ANTHROPIC_API_KEY"].strip()
    !pip -q install anthropic
    import anthropic
    client = anthropic.Anthropic()
    print("Real model mode")
else:
    print("Mock mode — deterministic rule-based stand-in (prompt text is ignored in this mode)")
print(nl.PROMPT_V2.format(company=COMPANY, description=DESCRIPTION, headline="<headline>", source="<source>", categories=nl.CATEGORIES))

Mock mode — deterministic rule-based stand-in (prompt text is ignored in this mode)
You are an analyst at a private equity firm screening news about Marriott (global hotel franchisor, NASDAQ: MAR).

For the headline below, return ONLY a JSON object with these keys:
- "relevant": true if the headline is about Marriott the company (its business, stock, management,
   properties, deals or regulation); false if it is about something else that merely shares the name
   (e.g. a person, a place, a different company) or is generic listicle/advertising content.
- "sentiment": one of "positive", "negative", "neutral" — from the perspective of an INVESTOR in
   Marriott. Ask: would this make an owner of the business more or less confident? Routine operational
   news (a hotel opening, a promotion) is "neutral" unless it signals a change in trajectory.
- "category": one of ['earnings', 'm&a_or_capital', 'management', 'demand_or_macro', 'legal_or_regulatory', 'product_or_brand', 'esg_or_labor', 'ot

In [14]:
# TODO: classify the frame with nl.classify_frame(df, COMPANY, DESCRIPTION, client=client) and look at the new llm_* columns

## Step 6 — Evaluate: lexicon vs LLM vs humans
Accuracy alone hides the interesting part. Look at the confusion matrix and the per-class report: *which* mistakes does each method make?

In [15]:
# TODO (if labels exist): merge llm columns onto m and compute
#   nl.accuracy(m.lex_label[rel], m.sentiment[rel]) and nl.accuracy(m.llm_sentiment[rel], m.sentiment[rel]) where rel = m.relevant
#   nl.accuracy(m.llm_relevant.astype(bool), m.relevant)
#   nl.confusion(...), nl.per_class_report(...)

In [16]:
# The most useful table in the whole exercise: where the model disagrees with the humans.
if "m" in globals() and "llm_sentiment" in m.columns:
    rel = m.relevant.astype(bool)
    dis = m[rel & (m.llm_sentiment != m.sentiment)][["title", "sentiment", "llm_sentiment", "llm_rationale"]]
    print(len(dis), "disagreements"); display(dis)
else:
    print("Finish the classification and evaluation TODOs above first.")

Finish the classification and evaluation TODOs above first.


## Step 7 — Iterate the prompt (REQUIRED — every team ships a V3)
1. **State the change before you run it.** Pick *one* thing V2 gets wrong (read your disagreement list), write one sentence: "V3 adds ___ because V2 ___".
2. Copy `PROMPT_V2` into `PROMPT_V3`, make exactly that one change (e.g. two labelled examples of the hard case, or a rule for 'cost cuts').
3. Re-run and re-score. **Log both numbers.** Keep V3 even if it is *worse* — say why. A logged regression teaches more than an unlogged win.

**Track A:** re-run `classify_frame(..., prompt_template=PROMPT_V3)`.
**Track B (mock mode ignores prompt text):** run V2 and V3 by hand in a chat UI on the same 20 headlines and score both against your labels — the discipline is identical.

In [17]:
# V3_CHANGE = "adds two labelled examples of cost-cut headlines, because V2 calls them negative and our gold says neutral"
# PROMPT_V3 = nl.PROMPT_V2 + "\n\nExamples:\n- 'Company cuts 200 corporate roles in efficiency push' -> sentiment neutral (cost discipline)\n"
# df_v3 = nl.classify_frame(df, COMPANY, DESCRIPTION, client=client, prompt_template=PROMPT_V3)
# TODO: re-compute accuracy on df_v3, then log:  print(f"V2: {acc_v2:.0%}  V3: {acc_v3:.0%}  change: {V3_CHANGE}")

## ⚡ Fast lanes (tech students: pick one while the team writes the debrief notes)
1. **Batching.** `classify_frame` calls the model once per headline. Batch 10 headlines per call (one JSON array out) and compare wall-clock time and accuracy.
2. **Model swap.** Re-run with a smaller/cheaper model (change the `model=` argument in `newslib.classify_headline`). Build a 2×2: accuracy vs cost.
3. **Cost tracker.** Count input/output tokens per call (the API response reports usage) and print cost-per-100-headlines.
4. **Dashboard.** Run the Streamlit app (cells below) — and deploy it on Streamlit Community Cloud if you finish early.

## Step 8 (fast lane) — Run the dashboard
`app/news_dashboard.py` is a Streamlit app that calls the same `newslib` functions. **Locally:** `streamlit run app/news_dashboard.py`. **In Colab:** run the two cells below — the first starts the server in the background, the second gives you a URL proxied through Colab (no third-party tunnel needed).

In [18]:
APP_DIR = "workshop/app" if os.path.exists("workshop/app") else "../app"
!pip -q install streamlit
# system_raw truly detaches the server. (A plain `!command &` hangs the cell:
# the server keeps the cell's terminal open and Colab waits for it to close.)
# CORS/XSRF off: Colab's proxy serves the page from a different origin, and
# with the protections on, Streamlit rejects the websocket — the page loads
# as empty grey blocks and never renders.
get_ipython().system_raw(
    f"cd {APP_DIR} && streamlit run news_dashboard.py "
    "--server.port 8501 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "> /tmp/streamlit.log 2>&1 &")
import time; time.sleep(5)
print(open("/tmp/streamlit.log").read()[-500:] if os.path.exists("/tmp/streamlit.log") else "server log not found — run this cell in Colab")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 75.6 MB/s eta 0:00:00


2026-10-18 16:24:48.448 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.12.126.225:8501




In [19]:
try:
    from google.colab.output import eval_js
    print("Open your dashboard:", eval_js("google.colab.kernel.proxyPort(8501)"))
except Exception:
    print("Not in Colab — open http://localhost:8501")

Open your dashboard: https://8501-m-s-kkb-euw4b0-1pt9q0smkpvw0-b.europe-west4-0.prod.colab.dev


## A note on version control (done on Day 1)
The finance cohort lived this loop yesterday with Case 1: they pushed `case1/whitespace.py` plus its tests to GitHub, watched Actions go green, broke a function on purpose, watched the red ✗ catch it, and fixed it. `app/newslib.py` follows exactly the same library-plus-tests pattern (its tests live in `app/tests/`), so the same safety net is one push away for today's prompt work: one repo per team, `app/` + `.github/` at the root, and every `PROMPT_V3` committed with its score in the message. Tech students — if a teammate's test idea captures a failure you found today, add it: `assert lexicon_sentiment("record impairment")["lex_label"] != "positive"` is the house style.

## Case 2 debrief prep (before 1:40)
Have ready: your accuracy table (lexicon vs LLM vs gold, with n), your agreement rate, one disagreement you found interesting, and your V2→V3 numbers with the one-sentence change rationale.